# 4. State, nodes and edges

**LangGraph tutorial, lesson 4 of 5**

**No tools in this notebook.** We are learning the three LangGraph nouns on
their own, because meeting them *and* tool calling simultaneously is what makes
LangGraph feel harder than it is.

| Noun | What it is |
|---|---|
| **State** | A dict every node reads from and writes to |
| **Node** | A plain function: takes state, returns the bits it changed |
| **Edge** | Which node runs next |

You declare nodes and edges, then `.compile()`. Compiling gives you something
with the same `.invoke()` interface as a model — which is why graphs nest neatly
inside other graphs.

## State, and the one genuinely unusual idea: reducers

By default a node returning `{"foo": x}` **replaces** `state["foo"]`.

For a message history, replacing is exactly wrong — we want to **append**. So
LangGraph lets you attach a *reducer*: a function describing how to merge a
node's output into existing state. `add_messages` appends instead of
overwriting.

> **Rule of thumb:** no annotation = overwrite, annotation = merge somehow.

`turn_count` below has no reducer, so you can watch the contrast.

In [ ]:
from typing import Annotated, TypedDict

from langchain_core.messages import AIMessage, HumanMessage
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages

from common import make_llm, text_of


class ChatState(TypedDict):
    messages: Annotated[list, add_messages]   # appended
    turn_count: int                           # overwritten

## Nodes

A node is just a function. No decorator, no base class. It receives the whole
current state and returns a dict of **only** the keys it wants to change.

In [ ]:
llm = make_llm()


def greet(state: ChatState) -> dict:
    """A node that only touches state -- it never calls a model."""
    print("  [node: greet] adding a framing instruction")
    return {
        "messages": [HumanMessage("Answer in exactly one short sentence.")],
        "turn_count": state.get("turn_count", 0) + 1,
    }


def call_model(state: ChatState) -> dict:
    """Send the whole history to the model, append the reply."""
    print(f"  [node: call_model] sending {len(state['messages'])} messages")
    reply = llm.invoke(state["messages"])
    # Returning a single message is fine -- add_messages wraps it in a list.
    return {"messages": [reply], "turn_count": state["turn_count"] + 1}

## Edges, then compile

`START` and `END` are sentinels marking where execution enters and leaves.

In [ ]:
builder = StateGraph(ChatState)

builder.add_node("greet", greet)
builder.add_node("call_model", call_model)

builder.add_edge(START, "greet")
builder.add_edge("greet", "call_model")
builder.add_edge("call_model", END)

graph = builder.compile()
print(graph.get_graph().draw_ascii())

## Run it

In [ ]:
result = graph.invoke(
    {
        "messages": [HumanMessage("Why are graphs a good fit for agent control flow?")],
        "turn_count": 0,
    }
)

print(f"turn_count: {result['turn_count']}   (overwritten on each write)")
print(f"messages  : {len(result['messages'])}   (appended on each write)\n")

for message in result["messages"]:
    who = "user" if isinstance(message, HumanMessage) else "assistant"
    print(f"  [{who}] {text_of(message)[:90]}")

## State does not survive between invocations

Each `.invoke()` starts clean. Nothing persists unless you attach a
*checkpointer* — that is the difference between a graph and a chat session.

In [ ]:
second = graph.invoke(
    {"messages": [HumanMessage("And what is a reducer?")], "turn_count": 0}
)
print(f"Second run message count: {len(second['messages'])} -- it did not remember run 1.")

## Recap

```
state  = TypedDict; reducers decide overwrite vs merge
node   = function(state) -> dict of changes
edge   = add_edge(from, to), with START and END as bookends
then   = .compile() -> something you can .invoke()
```

Every edge here was **fixed**: `greet` always leads to `call_model`. An agent
needs a *decision* — "tools, or done?" — which needs a **conditional edge**.

That is the last piece.

---
**Next:** `05_tool_agent.ipynb`